<a href="https://colab.research.google.com/github/Hale-T2/CHORD/blob/main/Chord_Estimation_RWC_P_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Block 1: Set up files and directories, ensure they exist**


In [ ]:
# Install pretty_midi for symbolic ingestion alongside standard MIR packages
!pip install -q pretty_midi librosa mido numpy scipy pandas matplotlib seaborn numba

# Mount Google Drive
from google.colab import drive
import os

if not os.path.exists('/content/gdrive'):
    drive.mount('/content/gdrive')

print("[SUCCESS] Environment initialized. Ready for RWC-R Audio & MIDI processing.")


base_gdrive_dir = "/content/gdrive/MyDrive/Colab Notebooks/rwc"
audio_dir = os.path.join(base_gdrive_dir, "RWC-P")
chord_dir = os.path.join(base_gdrive_dir, "chords", "RWC-P")
print(f"Audio Directory:     {audio_dir}")
print(f"CHORD Directory:      {chord_dir}")

# Perform a directory validation check
if os.path.exists(audio_dir) and os.path.exists(chord_dir):
    audio_count = len([f for f in os.listdir(audio_dir) if f.endswith('.wav')])
    chord_count = len([f for f in os.listdir(chord_dir) if f.endswith('.mid')])
    print(f"\n[+] Validation: Found {audio_count} WAV files and {chord_count} CHORD files.")
else:
    print("\n[-] Warning: Verify that 'Colab Notebooks/rwc/RWC-P' and 'Colab Notebooks/rwc/chord/RWC-P' are created in Google Drive.")



[SUCCESS] Environment initialized. Ready for RWC-R Audio & MIDI processing.
Audio Directory:     /content/gdrive/MyDrive/Colab Notebooks/rwc/RWC-P
CHORD Directory:      /content/gdrive/MyDrive/Colab Notebooks/rwc/chords/RWC-P

[+] Validation: Found 100 WAV files and 0 CHORD files.


## **Block 2: Acoustic Signal Preprocessing & Constant-Q Transform (CQT)**
The function compute_cqt_choromogram implements the signal-conditioning layer to perform **linear stereo-to-mono downmixing** and **signed 16-bit quantization normalization** ($x[n]/32768.0$) to avoid numerical overflow and spectral distortions in the logarithmic compression stage. We then compute the 12-dimensional Constant-Q Transform (CQT) chromagram.

In [ ]:
import numpy as np
import librosa
import soundfile as sf
from typing import Tuple

def compute_cqt_chromagram(audio_path: str, Fs: int = 22050, H: int = 2048) -> Tuple[np.ndarray, float]:
    """

    Loads audio, downmixes stereo to mono, normalizes quantization depth to float32 [-1.0, 1.0],

    and extracts an L2-normalized Constant-Q Transform (CQT) chromagram [4].

    """
    try:
        # Load using soundfile for explicit quantization handling (reads natively to float32)
        data, sr_native = sf.read(audio_path, dtype='float32')

        # 1. Stereo-to-Mono downmixing via arithmetic mean
        if len(data.shape) > 1 and data.shape[1] > 1:
            y_mono = np.mean(data, axis=1)
        else:
            y_mono = data

        # 2. Resampling to target rate (Fs = 22050 Hz)
        if sr_native != Fs:
            y = librosa.resample(y_mono, orig_sr=sr_native, target_sr=Fs)
        else:
            y = y_mono
    except Exception as e:
        # Robust fallback using librosa's loader pipeline
        y, sr_native = librosa.load(audio_path, sr=Fs, mono=True)
        y = y.astype(np.float32)

    # Compute CQT chroma representation (12 pitch-class bins)
    X = librosa.feature.chroma_cqt(y=y, sr=Fs, hop_length=H, norm=None)

    # Apply frame-wise L2 normalization to balance relative spectral energy
    norms = np.linalg.norm(X, axis=0, keepdims=True)
    norms[norms == 0] = 1.0
    X_norm = X / norms

    Fs_X = Fs / H  # Analysis feature rate (typically ~10.77 Hz)
    return X_norm, Fs_X

## **Block 3: Batch-Processing Google Drive (100 RWC tracks)**
This automated pipeline loops through your 100 tracks, loading `.wav` audio files and parsing `.csv` chord sheets stored in your Google Drive. It standardizes annotations into Christopher Harte syntax, extracts CQT features, computes frame-level predictions, and generates an aggregated macro performance report.

In [ ]:
import pandas as pd
from typing import List, Tuple

def convert_rwc_chord_label(label: str) -> str:
    """Standardizes RWC annotations into major/minor triads or 'N' (no chord)."""
    label = str(label).strip()
    if label in ['N', 'sil', 'pause', 'X', '', 'nan', '&pause']:
        return 'N'
    # Strip slash inversions
    if '/' in label:
        label = label.split('/')[0]
    # Extract root
    if len(label) >= 2 and label[1] in ['#', 'b']:
        root = label[0:2]
        rest = label[2:]
    else:
        root = label[0]
        rest = label[1:]
    # Enharmonic standardization (flat mapping to sharp)
    enharmonics = {'Db': 'C#', 'Eb': 'D#', 'Gb': 'F#', 'Ab': 'G#', 'Bb': 'A#'}
    if root in enharmonics:
        root = enharmonics[root]

    if rest.startswith(':'):
        rest = rest[1:]

    # Classify quality
    quality = 'm' if (rest.startswith('min') or rest.startswith('m') or rest.startswith('dim') or rest.startswith('hdim')) and not (rest.startswith('maj') or rest.startswith('major')) else ''
    final_label = f"{root}{quality}"

    chroma_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    valid_chords = chroma_labels + [c + 'm' for c in chroma_labels]

    return final_label if final_label in valid_chords else 'N'

def parse_csv_chord_file(csv_path: str) -> List[Tuple[float, float, str]]:
    """
    Parses a chord annotation file (CSV, LAB, or TXT) containing chord timing segments.
    Robustly handles semicolon (RWC standard), comma, tab, or space delimiters without crashing.
    """
    annotations = []
    try:
        with open(csv_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#') or line.startswith('t_start') or line.startswith('start_time'):
                    continue  # Skip comments and headers

                # 1. First Priority: Semicolon split (RWC Standard)
                parts = line.split(';')
                if len(parts) >= 3:
                    try:
                        start = float(parts[0].strip())
                        end = float(parts[1].strip())
                        label = ";".join(parts[2:]).strip()
                        label = label.replace('"', '').replace("'", "")
                        annotations.append((start, end, label))
                        continue
                    except ValueError:
                        pass

                # 2. Second Priority: Comma split (Standard CSV)
                parts = line.split(',')
                if len(parts) >= 3:
                    try:
                        start = float(parts[0].strip())
                        end = float(parts[1].strip())
                        label = ",".join(parts[2:]).strip()
                        label = label.replace('"', '').replace("'", "")
                        annotations.append((start, end, label))
                        continue
                    except ValueError:
                        pass

                # 3. Third Priority: Whitespace split (LAB/TXT format)
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        start = float(parts[0].strip())
                        end = float(parts[1].strip())
                        label = " ".join(parts[2:]).strip()
                        label = label.replace('"', '').replace("'", "")
                        annotations.append((start, end, label))
                    except ValueError:
                        continue
    except Exception as e:
        print(f"[-] Failed to parse file {os.path.basename(csv_path)}: {e}")
    return annotations

def compile_annotation_matrix(annotations: List[Tuple[float, float, str]], chord_labels: List[str], Fs_X: float, num_frames: int) -> np.ndarray:
    """Compiles segments into a frame-level binary matrix of shape (24, num_frames)."""
    ann_matrix = np.zeros((len(chord_labels), num_frames))
    for frame_idx in range(num_frames):
        frame_time = (frame_idx + 0.5) / Fs_X
        current_chord = 'N'
        for start, end, label in annotations:
            if start <= frame_time < end:
                current_chord = convert_rwc_chord_label(label)
                break
        if current_chord in chord_labels:
            ann_matrix[chord_labels.index(current_chord), frame_idx] = 1.0
    return ann_matrix


## **Block 4: Generate chord templates**
Function generate_triad_templates generates chord templates for the matching.



In [ ]:
def generate_triad_templates() -> np.ndarray:
    """Generates 12 major and 12 minor binary triad templates [41]."""
    template_cmaj = np.array([1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0]).T
    template_cmin = np.array([1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0]).T

    chord_templates = np.zeros((12, 24))
    for shift in range(12):
        chord_templates[:, shift] = np.roll(template_cmaj, shift)
        chord_templates[:, shift+12] = np.roll(template_cmin, shift)
    return chord_templates


## **Block 5: Template Matching Algorithm**
Function run_template_matching implements the template matching algorithm.

In [ ]:
def run_template_matching(X_chroma: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Executes deterministic CQT-Template Matching classifier."""
    templates = generate_triad_templates()
    templates_norm = templates.copy().astype(np.float64)
    t_norms = np.linalg.norm(templates_norm, axis=0, keepdims=True)
    t_norms[t_norms == 0] = 1.0
    templates_norm /= t_norms

    similarity = np.matmul(templates_norm.T, X_chroma)
    max_indices = np.argmax(similarity, axis=0)

    predictions = np.zeros(similarity.shape, dtype=np.int32)
    for f_idx, max_idx in enumerate(max_indices):
        predictions[max_idx, f_idx] = 1

    return similarity, predictions

## **Block 6: HMM Algorithm Setting up the HMM Transition Matrix**
Function learn_transition_matrix_from_data generates an HMM that is learned from its input gt_matrices: A list of 2D numpy arrays of shape (24, num_frames). It calls function apply_circular_diagonal_averaging in calculating the terms of the matrix.

Function generate_uniform_transition_matrix generates a uniform HMM matrix instead.

Function viterbi_log_likelihood implements the Viterbi algorithm for the HMM algorthm.

In [ ]:
import numpy as np
from numba import jit

def apply_circular_diagonal_averaging(A_raw: np.ndarray) -> np.ndarray:
    """
    Applies circular mean diagonal averaging to a 24x24 transition matrix
    to enforce key-invariance, as formulated in the paper.
    """
    A_ti = np.zeros_like(A_raw)

    # Define the boundaries for the four 12x12 sub-blocks:
    # (Major->Major, Major->Minor, Minor->Major, Minor->Minor)
    blocks = [
        (0, 12, 0, 12),
        (0, 12, 12, 24),
        (12, 24, 0, 12),
        (12, 24, 12, 24)
    ]

    for (r_start, r_end, c_start, c_end) in blocks:
        M = A_raw[r_start:r_end, c_start:c_end]
        M_ti = np.zeros((12, 12))

        # Calculate circular diagonal wrapping
        for i in range(12):
            for j in range(12):
                val = 0.0
                for k in range(12):
                    val += M[(i + k) % 12, (j + k) % 12]
                M_ti[i, j] = val / 12.0

        A_ti[r_start:r_end, c_start:c_end] = M_ti

    return A_ti

def learn_transition_matrix_from_data(gt_matrices: list, num_states: int = 24, apply_shift_invariance: bool = True, epsilon: float = 1e-6) -> np.ndarray:
    """
    Learns a Markov transition matrix from a list of ground-truth matrices.

    Args:
        gt_matrices: A list of 2D numpy arrays of shape (24, num_frames).
        num_states: The total number of hidden chord states.
        apply_shift_invariance: Whether to apply circular diagonal averaging.
        epsilon: A small Laplace smoothing factor to prevent zero probabilities.
    """
    # Initialize with Laplace smoothing
    transition_counts = np.ones((num_states, num_states)) * epsilon

    for gt_matrix in gt_matrices:
        # Check if the frame contains a valid chord (non-silent)
        col_sums = np.sum(gt_matrix, axis=0)

        prev_state = -1
        for frame_idx in range(gt_matrix.shape[1]):
            if col_sums[frame_idx] > 0.0:
                # Get the index of the active chord (0 to 23)
                curr_state = int(np.argmax(gt_matrix[:, frame_idx]))

                # Register the transition if a previous valid state exists
                if prev_state != -1:
                    transition_counts[prev_state, curr_state] += 1.0

                prev_state = curr_state
            else:
                # Reset transition tracker on silence/No Chord
                prev_state = -1

    # Apply circular diagonal wrapping to block matrices
    if apply_shift_invariance and num_states == 24:
        transition_counts = apply_circular_diagonal_averaging(transition_counts)

    # Row-normalize to convert counts into standard probabilities
    row_sums = np.sum(transition_counts, axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0 # Prevent division by zero
    A_learned = transition_counts / row_sums

    return A_learned


def generate_uniform_transition_matrix(self_transition_prob: float = 0.90, num_states: int = 24) -> np.ndarray:
    """Creates transition grid with diagonal self-loop bias."""
    off_diagonal_prob = (1.0 - self_transition_prob) / (num_states - 1)
    A = off_diagonal_prob * np.ones((num_states, num_states))
    np.fill_diagonal(A, self_transition_prob)
    return A

@jit(nopython=True)
def viterbi_log_likelihood(A: np.ndarray, C: np.ndarray, B_O: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Viterbi algorithm in log-space to bypass numerical underflow [49]."""
    I = A.shape[0]
    N = B_O.shape[1]
    tiny = np.finfo(0.).tiny

    A_log = np.log(A + tiny)
    C_log = np.log(C + tiny).flatten()
    B_O_log = np.log(B_O + tiny)

    D_log = np.zeros((I, N))
    E = np.zeros((I, N-1)).astype(np.int32)
    D_log[:, 0] = C_log + B_O_log[:, 0]

    for n in range(1, N):
        for i in range(I):
            temp_sum = A_log[:, i] + D_log[:, n-1]
            D_log[i, n] = np.max(temp_sum) + B_O_log[i, n]
            E[i, n-1] = np.argmax(temp_sum)

    S_opt = np.zeros(N).astype(np.int32)
    S_opt[-1] = np.argmax(D_log[:, -1])
    for n in range(N-2, -1, -1):
        S_opt[n] = E[int(S_opt[n+1]), n]

    S_mat = np.zeros((I, N)).astype(np.int32)
    for n in range(N):
        S_mat[S_opt[n], n] = 1

    return S_mat, S_opt

## **Block 7: MAIN code**
Implement the code

In [ ]:
import scipy.io.wavfile as wavfile
from torch.utils.data import DataLoader

if not os.path.exists(audio_dir) or not os.path.exists(chord_dir):
    print("[-] Google Drive paths do not exist. Please mount drive or check directory paths.")

chroma_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
chord_labels = chroma_labels + [c + 'm' for c in chroma_labels]

global_tp_tmpl, global_fp_tmpl, global_fn_tmpl = 0, 0, 0
global_tp_hmm, global_fp_hmm, global_fn_hmm = 0, 0, 0
track_reports = []
# Caching lists for the two-pass architecture
cached_features = []
all_gt_matrices = []

print("="*80)
print("        RWC 100-TRACK POPULAR MUSIC BENCHMARK SUITE (MIR STANDARDS)")
print("="*80)
print("[*] PASS 1: Extracting CQT features and compiling CSV annotations...")

# ==========================================================
# PASS 1: Data Extraction & Ground Truth Compilation
# ==========================================================
for i in range(1, 101):
    # Accommodate padding and variable naming styles (e.g. RWC_P001.wav vs RWC_001.wav, etc.)
    wav_name = f"RWC_P{i:03d}.wav"
    chord_name = f"RWC_P{i:03d}.csv"

    audio_path = os.path.join(audio_dir, wav_name)
    chord_path = os.path.join(chord_dir, chord_name)

    # Handle alternative unpadded/stripped filenames if needed
    if not os.path.exists(audio_path):
        alt_wav = f"RWC_P{i:03d}.wav"
        if os.path.exists(os.path.join(audio_dir, alt_wav)):
            audio_path = os.path.join(audio_dir, alt_wav)
        else:
            print(f"[-] Track {i:03d}: Missing files. Skipped.")
            continue

    if not os.path.exists(chord_path):
        alt_chord = f"RWC_P{i:03d}.csv"
        if os.path.exists(os.path.join(chord_dir, alt_chord)):
            chord_path = os.path.join(chord_dir, alt_chord)
        else:
            print(f"[-] Track {i:03d}: Missing chord csv file. Skipped.")
            continue

    try:
        # 1. CQT Chromagram Feature Extraction
        X_audio, Fs_X = compute_cqt_chromagram(audio_path)
        num_frames = X_audio.shape[1]

        # 2. Parse Ground Truth and Map to Frame-Level Chronology
        parsed_ann = parse_csv_chord_file(chord_path)
        gt_matrix = compile_annotation_matrix(parsed_ann, chord_labels, Fs_X, num_frames)

        # Cache the data for Pass 2 to avoid re-extracting audio
        cached_features.append((i, X_audio, gt_matrix))
        all_gt_matrices.append(gt_matrix)
    except Exception as e:
        print(f"[!] Track {i:03d} Extraction Failed: {e}")
# ==========================================================
# INTERMEDIATE STEP: Learn Markov Matrix
# ==========================================================
print(f"[*] Compiling dynamic transition matrix from {len(all_gt_matrices)} tracks...")
learned_A = learn_transition_matrix_from_data(all_gt_matrices, apply_shift_invariance=True)

print("[*] PASS 2: Executing Inference and Viterbi Decoding...")

# ==========================================================
# PASS 2: Model Inference & Evaluation
# ==========================================================
for i, X_audio, gt_matrix in cached_features:
    try:
        num_frames = X_audio.shape[1]
        # 4. Predict Chords - Stateless Template Matching
        similarity, preds_tmpl = run_template_matching(X_audio)

        # 5. Predict Chords - Temporal Log-Scaled HMM (Viterbi)
        A = learned_A
        C = (1.0 / 24.0) * np.ones(24)

        # Project similarity to frame-level likelihood distribution
        # Cosine similarity range [-1.0, 1.0] scaled using soft-max
        templates = generate_triad_templates()
        templates_norm = templates.copy().astype(np.float64)
        t_norms = np.linalg.norm(templates_norm, axis=0, keepdims=True)
        t_norms[t_norms == 0] = 1.0
        templates_norm /= t_norms

        sim_scores = np.matmul(templates_norm.T, X_audio)
        B_emission = np.exp(sim_scores * 5.0)  # Scaling factor gamma=5.0
        B_emission /= np.sum(B_emission, axis=0, keepdims=True)

        preds_hmm, _ = viterbi_log_likelihood(A, C, B_emission)

        # 6. Accumulate Metrics - Template Matching
        tp_t = int(np.sum(gt_matrix * preds_tmpl))
        fp_t = int(np.sum((1 - gt_matrix) * preds_tmpl))
        fn_t = int(np.sum(gt_matrix * (1 - preds_tmpl)))

        p_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0.0
        r_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0
        f1_t = 2 * p_t * r_t / (p_t + r_t) if (p_t + r_t) > 0 else 0.0

        global_tp_tmpl += tp_t
        global_fp_tmpl += fp_t
        global_fn_tmpl += fn_t

        # 7. Accumulate Metrics - Temporal HMM
        tp_h = int(np.sum(gt_matrix * preds_hmm))
        fp_h = int(np.sum((1 - gt_matrix) * preds_hmm))
        fn_h = int(np.sum(gt_matrix * (1 - preds_hmm)))

        p_h = tp_h / (tp_h + fp_h) if (tp_h + fp_h) > 0 else 0.0
        r_h = tp_h / (tp_h + fn_h) if (tp_h + fn_h) > 0 else 0.0
        f1_h = 2 * p_h * r_h / (p_h + r_h) if (p_h + r_h) > 0 else 0.0

        global_tp_hmm += tp_h;
        global_fp_hmm += fp_h
        global_fn_hmm += fn_h

        track_reports.append({
            "Track": f"RWC_R{i:03d}",
            "TP": tp_t, "FP": fp_t, "FN": fn_t, "Tmpl_P": p_t, "Tmpl_R": r_t, "Tmpl_F1": f1_t,
            "TP": tp_h, "FP": fp_h, "FN": fn_h, "HMM_P": p_h, "HMM_R": r_h, "HMM_F1": f1_h
        })

#        print(f"[+] Track {i:02d} Processed. F1-Score: Template = {f1_t:.4f} | HMM = {f1_h:.4f}")
        print(f"[+] Track {i:02d}. Template: TP={tp_t:.4f} FP={fp_t:.4f} FN={fn_t:.4f} P={p_t:.4f} R={r_t:.4f} F1={f1_t:.4f} | HMM: TP={tp_h:.4f} FP={fp_h:.4f} FN={fn_h:.4f} P={p_h:.4f} R={r_h:.4f} F1={f1_h:.4f}")
    except Exception as e:
        print(f"[!] Failed processing Track {i:03d}: {e}")

# Compile and save macro metrics
if len(track_reports) > 0:
    macro_p_t = global_tp_tmpl / (global_tp_tmpl + global_fp_tmpl) if (global_tp_tmpl + global_fp_tmpl) > 0 else 0.0
    macro_r_t = global_tp_tmpl / (global_tp_tmpl + global_fn_tmpl) if (global_tp_tmpl + global_fn_tmpl) > 0 else 0.0
    macro_f1_t = 2 * macro_p_t * macro_r_t / (macro_p_t + macro_r_t) if (macro_p_t + macro_r_t) > 0 else 0.0

    macro_p_h = global_tp_hmm / (global_tp_hmm + global_fp_hmm) if (global_tp_hmm + global_fp_hmm) > 0 else 0.0
    macro_r_h = global_tp_hmm / (global_tp_hmm + global_fn_hmm) if (global_tp_hmm + global_fn_hmm) > 0 else 0.0
    macro_f1_h = 2 * macro_p_h * macro_r_h / (macro_p_h + macro_r_h) if (macro_p_h + macro_r_h) > 0 else 0.0

    report_path = os.path.join(base_gdrive_dir, "rwc_p-2_evaluation_report.csv")
    pd.DataFrame(track_reports).to_csv(report_path, index=False)

    print("\n" + "="*80)
    print("                     AGGREGATED RWC-P MACRO PERFORMANCE                     ")
    print("="*80)
    print(f"Model Class        | Precision | Recall   | F1-Score")
    print(f"----------------------------------------------------")
    print(f"CQT-Template       | {macro_p_t:.4f}    | {macro_r_t:.4f}   | {macro_f1_t:.4f}")
    print(f"CQT-HMM (Viterbi)  | {macro_p_h:.4f}    | {macro_r_h:.4f}   | {macro_f1_h:.4f}")
    print(f"----------------------------------------------------")
    print(f"[SAVED] Comparative performance report exported: {report_path}")
    print("="*80)

        RWC 100-TRACK POPULAR MUSIC BENCHMARK SUITE (MIR STANDARDS)
[*] PASS 1: Extracting CQT features and compiling CSV annotations...
[*] Compiling dynamic transition matrix from 100 tracks...
[*] PASS 2: Executing Inference and Viterbi Decoding...
[+] Track 01. Template: TP=649.0000 FP=1582.0000 FN=1549.0000 P=0.2909 R=0.2953 F1=0.2931 | HMM: TP=827.0000 FP=1404.0000 FN=1371.0000 P=0.3707 R=0.3763 F1=0.3734
[+] Track 02. Template: TP=553.0000 FP=1823.0000 FN=1680.0000 P=0.2327 R=0.2476 F1=0.2400 | HMM: TP=548.0000 FP=1828.0000 FN=1685.0000 P=0.2306 R=0.2454 F1=0.2378
[+] Track 03. Template: TP=948.0000 FP=1134.0000 FN=1065.0000 P=0.4553 R=0.4709 F1=0.4630 | HMM: TP=1132.0000 FP=950.0000 FN=881.0000 P=0.5437 R=0.5623 F1=0.5529
[+] Track 04. Template: TP=670.0000 FP=1920.0000 FN=1883.0000 P=0.2587 R=0.2624 F1=0.2605 | HMM: TP=1122.0000 FP=1468.0000 FN=1431.0000 P=0.4332 R=0.4395 F1=0.4363
[+] Track 05. Template: TP=1094.0000 FP=1341.0000 FN=1301.0000 P=0.4493 R=0.4568 F1=0.4530 | HMM